## Data processing and EDA 

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

1 Merge two dataset into one 

Loads the customers.json and transaction.json files, then joins them together by matching the transaction's sender account to the customer record. The combined result is saved to data.json.

In [2]:
cust = pd.read_json('../data/customers.json')
trans = pd.read_json('../data/transaction.json')

df = pd.merge(trans,cust, left_on= 'Sender Account ID', right_on = 'Customer ID', how = "left")
df = df.drop(columns = ['Customer ID'])
df = df.fillna(0)

df.to_json('../data/data_2/data.json')


Visualization 


+ Box plot

    The data appears to be synthetic or mock data. The lack of variance between categories like "Student" and "Freelancer," or "Starbucks" and "Monthly Salary," suggests that the variables in your dataset (Gender, Working Status, etc.) currently have no correlation with the Amounts or Balances.


In [3]:
import os
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend (no display needed)
import matplotlib.pyplot as plt

In [4]:
pd.read_json('../data/data_2/data.json')
DATA_PATH = os.path.join(os.getcwd(), '..', 'data', 'data_2', 'data.json')
df = pd.read_json(DATA_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Boxplot Analysis - Data Distribution & Outliers", fontsize=16, fontweight='bold')

# 3. Plot 1: Transaction Amount by Type
sns.boxplot(data=df, x='Transaction Detail', y='Transaction amount', hue='Transaction Detail', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title("Amount by Transaction Type")
axes[0].tick_params(axis='x', rotation=40)

# 4. Plot 2: Transaction Amount by Gender
sns.boxplot(data=df, x='Gender', y='Transaction amount', hue='Gender', ax=axes[1], palette='pastel', legend=False)
axes[1].set_title("Amount by Gender")

# 5. Plot 3: Account Balance by Working Status
sns.boxplot(data=df, x='Working Status', y='Account balance', hue='Working Status', ax=axes[2], palette='muted', legend=False)
axes[2].set_title("Balance by Working Status")
axes[2].tick_params(axis='x', rotation=40)

# 6. Save the image and display it
plt.tight_layout()
output_path = os.path.join(os.getcwd(), 'boxplot_output.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Boxplot saved to: {output_path}")

Boxplot saved to: c:\Users\Admin\Documents\group-project-spr-2026-grp2_2\backend\ai\ML\notebook\boxplot_output.png


+ Heat Map
    This confirms the data is likely synthetic and generated using independent random distributions for each column. The columns don't "talk" to each other.

In [5]:
import os
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [6]:
# 1. Load the dataset directly
DATA_PATH = os.path.join(os.getcwd(), '..', 'data', 'data_2', 'data_encoded.json')
df = pd.read_json(DATA_PATH)

# 2. Isolate numerical columns and filter out 'ID' columns
numerical_cols = df.select_dtypes(include='number')
cols_to_keep = [c for c in numerical_cols.columns if 'id' not in c.lower()]
df_numeric = numerical_cols[cols_to_keep]

# 3. Compute the correlation matrix
correlation_matrix = df_numeric.corr()

# 4. Set up the canvas and draw the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,          # Show the correlation number
    fmt=".2f",           # Round to 2 decimal places
    cmap="coolwarm",     # Blue = negative, Red = positive
    center=0,            # Center the color scale at 0
    linewidths=0.5,      # Add gridlines for readability
    square=True          # Force cells to be square
)

# 5. Format titles/labels, save, and display
plt.title("Feature Correlation Heatmap", fontsize=16, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right') # Tilt x-axis labels so they don't overlap
plt.tight_layout()

output_path = os.path.join(os.getcwd(), 'heatmap_output.png')
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Heatmap saved to: {output_path}")

Heatmap saved to: c:\Users\Admin\Documents\group-project-spr-2026-grp2_2\backend\ai\ML\notebook\heatmap_output.png


2 Preprocessing
   
In this part, we will provide more feautures for learning, and some are normalized, encoded

2.1 Extract Features

Creates new, more useful columns from existing data:

Age: Calculates age from the customer's Date of Birth.

Is_Weekend: A 1/0 flag for whether the transaction happened on a weekend.

Is_Night: A 1/0 flag for whether the transaction happened between 10 PM and 6 AM.

Balance_to_Salary_Ratio: How big the account balance is compared to monthly salary.

Transaction_to_Balance_Ratio: What fraction of the account balance was spent in this transaction.

In [7]:
if 'Date of Birth' in df.columns:
    df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
    df['Age'] = (datetime(2026, 3, 6) - df['Date of Birth']).dt.days // 365
    df = df.drop(columns=['Date of Birth'])

#Time-based flags
if 'DayofWeek' in df.columns:
    df['Is_Weekend'] = df['DayofWeek'].isin([5,6]).astype(int)
if 'Hours' in df.columns:
    df['Is_Night'] = df['Hour'].apply(lambda h: 1 if h >= 22 or h <= 6 else 0)
    
# Ratio Features
if 'Account Balance' in df.columns and 'Salary (per month)' in df.columns:
    salary = df['Salary (per month)'].replace(0, np.nan)
    df['Balance_to_Salary_Ratio'] = df['Account balance'] / salary
if 'Transaction amount' in df.columns and 'Account balance' in df.columns:
    balance = df['Account balance'].replace(0, np.nan)
    df['Transaction_to_Balance_Ratio'] = df['Transaction amount'] / balance

df = df.fillna(0)

2.2 Normalize

Rescales large number columns (like Transaction amount and Account balance) to a 0–1 range, so that the models don't incorrectly treat larger numbers as more important than smaller ones.

In [8]:
cols_to_scale = [
    'Transaction amount', 'Account balance', 'Salary (per month)',
    'Age', 'Balance_to_Salary_Ratio', 'Transaction_to_Balance_Ratio',
]
available = [c for c in cols_to_scale if c in df.columns]

scaler = MinMaxScaler()
df[available] = scaler.fit_transform(df[available])

3 Encoding

In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

3.1 Utility Functions

We'll keep your core logic here. Converts text columns (like "Male" or "Online Shopping") into numbers, since machine learning models can only work with numbers, not text.


In [10]:
def encode_columns(df, columns=None):
    if columns is None:
        columns = df.select_dtypes(include=['object']).columns.tolist()

    encoders = {}
    for col in columns:
        if col in df.columns:
            encoder = LabelEncoder()
            # We cast to string to avoid errors with mixed types or NaNs
            df[col] = encoder.fit_transform(df[col].astype(str))
            encoders[col] = encoder
            print(f"Successfully encoded: {col}")

    return df, encoders

3.2 Configuration & File Paths

Sets up the input and output file paths and defines which text columns need to be encoded (e.g., Gender, Location, Working Status).

In [11]:
# Paths
INPUT_FILE = '../data/data_2/data.json'
OUTPUT_FILE = '../data/data_2/data_encoded.json'

# Columns to encode (text → numbers)
TEXT_COLUMNS = [
    'Transaction Detail', 'Geological', 'Device Use', 
    'Gender', 'Location', 'Working Status'
]

3.3 Execution & Data Verification

Loads the data, applies the encoding, saves the result to data_encoded.json, and previews the first few rows to confirm everything looks correct.


In [12]:
# 1. Load data
if os.path.exists(INPUT_FILE):
    df = pd.read_json(INPUT_FILE)
    print(f"Loaded {len(df)} rows.")
    
    # 2. Encode
    df_encoded, encoders = encode_columns(df.copy(), TEXT_COLUMNS)
    
    # 3. Save
    df_encoded.to_json(OUTPUT_FILE, orient='records', indent=4, force_ascii=False)
    print(f"Saved encoded data to: {OUTPUT_FILE}")
    
    # 4. Preview the transformation
    display(df_encoded.head())
else:
    print(f"Error: Could not find file at {INPUT_FILE}. Please check your path!")

Loaded 25000 rows.
Successfully encoded: Transaction Detail
Successfully encoded: Geological
Successfully encoded: Device Use
Successfully encoded: Gender
Successfully encoded: Location
Successfully encoded: Working Status
Saved encoded data to: ../data/data_2/data_encoded.json


,Transaction ID,Sender Account ID,Receiver Account ID,Transaction amount,Timestamp,Transaction Detail,Geological,Device Use,Date of Birth,Gender,Location,Account balance,Transaction Count,Working Status,Salary (per month)
0,TXN_300001,ACC_07769,REC_1837,445095,2025-06-22 22:26:00,6,2,1,1967-09-19,0,4,195000000,4,4,48000000
1,TXN_300002,ACC_00215,REC_4611,4021124,2025-09-20 08:19:00,4,2,4,2004-09-04,0,5,276000000,2,4,35000000
2,TXN_300003,ACC_02078,REC_2498,4379954,2025-12-28 10:28:00,5,2,1,1965-05-05,1,4,387000000,3,5,10000000
3,TXN_300004,ACC_09065,REC_2911,5868893,2025-04-20 14:26:00,0,0,1,1978-07-28,2,4,464000000,5,4,46000000
4,TXN_300005,ACC_06874,REC_5761,4054613,2025-03-26 11:49:00,2,2,2,1994-05-04,0,4,272000000,6,4,13000000
